# Databricks to MongoDB Connectivity Check

Use this notebook to diagnose and verify connectivity from a Databricks cluster to a MongoDB replica set.

**Connection details (from your environment):**
- Hosts: `HBXPRESSFRMDRDB1.hbctxdom.com:28181`, `HBXPRESSFRMPRDB1.hbctxdom.com:28181`, `HBXPRESSFRMPRDDB1.hbctxdom.com:28181`
- Port: `28181`
- Replica Set: `xpressforms`
- TLS: Enabled (`tlsCAFile=/etc/pki/certs/ca.cert`)
- Auth Source: `servicerequest`
- IP Addresses: `10.227.12.164`, `10.226.109.26`, `10.225.74.130`

## Step 1 — Network Reachability (Port Check)

Check whether the Databricks cluster can reach MongoDB on port 28181 using `nc` (netcat).

In [ ]:
import subprocess

hosts = [
    ("HBXPRESSFRMDRDB1.hbctxdom.com", 28181),
    ("HBXPRESSFRMPRDB1.hbctxdom.com",  28181),
    ("HBXPRESSFRMPRDDB1.hbctxdom.com", 28181),
]

for host, port in hosts:
    result = subprocess.run(
        ["nc", "-zv", "-w", "5", host, str(port)],
        capture_output=True, text=True
    )
    status = "REACHABLE" if result.returncode == 0 else "UNREACHABLE"
    print(f"{host}:{port} -> {status}")
    if result.stderr:
        print(f"  {result.stderr.strip()}")

## Step 2 — DNS Resolution

Verify that the Databricks cluster can resolve the MongoDB hostnames to the expected IP addresses.

In [ ]:
import socket

hostnames = [
    "HBXPRESSFRMDRDB1.hbctxdom.com",
    "HBXPRESSFRMPRDB1.hbctxdom.com",
    "HBXPRESSFRMPRDDB1.hbctxdom.com",
]

expected_ips = {"10.227.12.164", "10.226.109.26", "10.225.74.130"}

for hostname in hostnames:
    try:
        resolved_ip = socket.gethostbyname(hostname)
        match = "OK" if resolved_ip in expected_ips else "UNEXPECTED IP"
        print(f"{hostname} -> {resolved_ip}  [{match}]")
    except socket.gaierror as e:
        print(f"{hostname} -> DNS RESOLUTION FAILED: {e}")

## Step 3 — TLS Certificate Check

Verify the CA certificate file exists on the Databricks cluster nodes.

> **Note:** The CA cert path `/etc/pki/certs/ca.cert` must be present on every worker node.
> If it is missing, copy it via an init script or store it in DBFS.

In [ ]:
import os

ca_cert_path = "/etc/pki/certs/ca.cert"

if os.path.exists(ca_cert_path):
    size = os.path.getsize(ca_cert_path)
    print(f"CA certificate found: {ca_cert_path}  ({size} bytes)")
else:
    print(f"CA certificate NOT FOUND at: {ca_cert_path}")
    print()
    print("Fix options:")
    print("  1. Upload the cert to DBFS: /dbfs/mnt/certs/ca.cert")
    print("  2. Reference it in the connection string as: tlsCAFile=/dbfs/mnt/certs/ca.cert")
    print("  3. Install via a cluster init script that copies the cert on startup")

## Step 4 — Install pymongo

Install the MongoDB Python driver on the cluster.

In [ ]:
%pip install pymongo dnspython

## Step 5 — Test MongoDB Connection (pymongo)

Attempt an actual connection to the replica set using the connection string from your environment.

> Replace `YOUR_PASSWORD` with the actual password (use Databricks Secrets to avoid hardcoding).

In [ ]:
from pymongo import MongoClient
from pymongo.errors import ConnectionFailure, ServerSelectionTimeoutError

# --- Use Databricks Secrets to avoid hardcoding credentials ---
# username = dbutils.secrets.get(scope="mongo-scope", key="username")
# password = dbutils.secrets.get(scope="mongo-scope", key="password")

username = "xpreaduser"
password = "YOUR_PASSWORD"          # replace or use dbutils.secrets above
ca_cert  = "/etc/pki/certs/ca.cert" # adjust path if cert is on DBFS

connection_string = (
    f"mongodb://{username}:{password}"
    "@HBXPRESSFRMDRDB1.hbctxdom.com:28181"
    ",HBXPRESSFRMPRDB1.hbctxdom.com:28181"
    ",HBXPRESSFRMPRDDB1.hbctxdom.com:28181"
    f"/?replicaSet=xpressforms"
    f"&tls=true"
    f"&tlsCAFile={ca_cert}"
    "&authSource=servicerequest"
)

try:
    client = MongoClient(connection_string, serverSelectionTimeoutMS=10000)
    # ping forces an actual network round-trip
    client.admin.command("ping")
    print("SUCCESS: Connected to MongoDB replica set")
    print(f"Server info: {client.server_info()}")
except ServerSelectionTimeoutError as e:
    print(f"TIMEOUT: Could not reach MongoDB within 10 s\n  {e}")
except ConnectionFailure as e:
    print(f"CONNECTION FAILED: {e}")
except Exception as e:
    print(f"ERROR: {type(e).__name__}: {e}")
finally:
    try:
        client.close()
    except Exception:
        pass

## Step 6 — List Databases (sanity check after connection)

If Step 5 succeeded, this confirms read access.

In [ ]:
from pymongo import MongoClient

client = MongoClient(connection_string, serverSelectionTimeoutMS=10000)

try:
    db_list = client.list_database_names()
    print("Databases visible to this user:")
    for db in db_list:
        print(f"  - {db}")
except Exception as e:
    print(f"Could not list databases: {e}")
finally:
    client.close()

## Troubleshooting Reference

| Symptom | Likely Cause | Fix |
|---|---|---|
| `UNREACHABLE` in Step 1 | Firewall / VPC peering not configured | Open port 28181 from Databricks subnet to MongoDB IPs in your network ACL / security group |
| `DNS RESOLUTION FAILED` in Step 2 | Custom DNS not configured on Databricks VPC | Add the `hbctxdom.com` DNS server to your VPC resolver, or add static `/etc/hosts` entries via init script |
| `CA certificate NOT FOUND` in Step 3 | Cert not deployed to cluster nodes | Upload cert to DBFS and update `tlsCAFile` path in the connection string |
| `ServerSelectionTimeoutError` in Step 5 | Port/firewall issue OR wrong replica set name | Double-check port 28181 is open; verify `replicaSet=xpressforms` |
| `Authentication failed` in Step 5 | Wrong password or `authSource` | Confirm password and that `authSource=servicerequest` is correct |
| `SSL: CERTIFICATE_VERIFY_FAILED` | CA cert mismatch or wrong path | Ensure the CA cert on the cluster was issued by the same CA that signed the MongoDB server cert |